In [37]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [38]:
ds = load_dataset("christinacdl/binary_hate_speech")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    3883 non-null   object
 1   label   3883 non-null   object
dtypes: object(2)
memory usage: 60.8+ KB


In [39]:
test['label'] = test['label'].apply(lambda x: 'hateful' if x == 'OFF_HATEFUL_TOXIC' else 'safe')

labels = test['label'].unique()

test

,text,label
0,i have to study... #face #pizza (i stole my ...,safe
1,days porn movie srilankanboyssex,safe
2,feeling for friends left in the place we use...,safe
3,why only target little #muslim children for mi...,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe
...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe
3880,A hoe wants attention a women wants respect.,hateful
3881,@user there is only one requirement for the jo...,hateful


In [40]:
load_dotenv()

api_key=os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [41]:
def classify(text, labels):

    sys_instruct="You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."

    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_CIVIC_INTEGRITY",
                threshold="BLOCK_NONE"
            ),
            ],
        ),
        contents=f"Classify the following text based on the task: Sentiment analysis of possibly hateful content. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Tweet: {text}"
    )

    request_time = time.time() - start_time
    completion = response.text
    if completion:
        completion = completion.lower()
    else:
        completion = "None"
    completion_tokens = response.usage_metadata.candidates_token_count
    prompt_tokens = response.usage_metadata.prompt_token_count
    total_tokens = response.usage_metadata.total_token_count

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'safe' in text:
        return 'safe'
    elif 'hateful' in text:
        return 'hateful'
    else:
        return 'error'

In [42]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_gemini_ZS_binary1.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/gemini_ZS_binary1.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,i have to study... #face #pizza (i stole my ...,safe,safe\n,1.135128,2.0,100.0,102.0,safe
1,days porn movie srilankanboyssex,safe,hateful\n,1.084316,3.0,86.0,89.0,hateful
2,feeling for friends left in the place we use...,safe,safe\n,1.058214,2.0,104.0,106.0,safe
3,why only target little #muslim children for mi...,hateful,hateful\n,1.106754,3.0,104.0,107.0,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe,safe\n,1.066569,2.0,128.0,130.0,safe
...,...,...,...,...,...,...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful,hateful\n,2.445403,3.0,105.0,108.0,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe,hateful\n,1.047649,3.0,113.0,116.0,hateful
3880,A hoe wants attention a women wants respect.,hateful,hateful\n,1.064875,3.0,86.0,89.0,hateful
3881,@user there is only one requirement for the jo...,hateful,safe\n,1.073808,2.0,109.0,111.0,safe


In [43]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,i have to study... #face #pizza (i stole my ...,safe,safe\n,1.135128,2.0,100.0,102.0,safe
1,days porn movie srilankanboyssex,safe,hateful\n,1.084316,3.0,86.0,89.0,hateful
2,feeling for friends left in the place we use...,safe,safe\n,1.058214,2.0,104.0,106.0,safe
3,why only target little #muslim children for mi...,hateful,hateful\n,1.106754,3.0,104.0,107.0,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe,safe\n,1.066569,2.0,128.0,130.0,safe
...,...,...,...,...,...,...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful,hateful\n,2.445403,3.0,105.0,108.0,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe,hateful\n,1.047649,3.0,113.0,116.0,hateful
3880,A hoe wants attention a women wants respect.,hateful,hateful\n,1.064875,3.0,86.0,89.0,hateful
3881,@user there is only one requirement for the jo...,hateful,safe\n,1.073808,2.0,109.0,111.0,safe


In [44]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.714911
F1 score: 0.715091
Precision: 0.716167
Recall: 0.714911


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [45]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.9512194884121188
Average completion tokens: 2.475122454240784
Average prompt tokens: 106.9943342776204
Average total tokens: 109.46690703064641


In [60]:
pred_df['completion_tokens'] = pred_df['completion_tokens'].fillna(0)

In [61]:
input_token_price = 0.1/1_000_000
output_token_price = 0.4/1_000_000

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.04538629999999995


In [62]:
with open('results/gemini_ZS_binary1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')